# R-GCN node prediction: rank reification at K = 2

**Question.** Does storing the rank behind an extra node (`rank_reification`) explain why Hybrid needs one more
layer than Ad-hoc in node prediction? In the K-sweep (`RGCN_k_sweep.ipynb`), R-GCN Hybrid scored 0.827 test ROC-AUC
at K = 2 against Ad-hoc's 0.869, and the two were equal from K = 3 on. An entry's own grid rank is 2 hops away in
Ad-hoc (entry → Position → rank) and 3 in Hybrid (entry → Position → `Position/<n>/hasRank` → rank), so a
2-layer Hybrid model cannot read it. At K = 3 the enrichment ablation found no unit that matters
(`RGCN_ablation.ipynb`), which is expected when both paths are in reach. Here the same rank unit is ablated at K = 2.

**Prediction (stated before the run):** `hybrid_minus_rank_reification` rises to about Ad-hoc's K = 2 level, and
`adhoc_plus_rank_reification` drops to about Hybrid's. No other unit changes the rank's distance, so the
other eight variants are not run.

**Protocol: the enrichment ablation's**, at K = 2. Every variant gets the Hybrid winner's configuration (dim, bases,
lr, wd) with one layer fewer (`hidden_list = [dim]`). Each seed in `FINAL_SEEDS` early-stops on val after the
50-epoch warm-up, and its best-val checkpoint is scored on test (`run_ablation_variant`). The two controls are
re-run at K = 2. The K = 3 rows of the same four graphs are read from the ablation for comparison. No shuffled-label
check: these are the ablation's graphs, which passed it there (the check tests the labels, not the depth).

In [1]:
# Must run before CUDA is initialised: avoids fragmentation OOMs across a long sweep.
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

import sys
from pathlib import Path

ROOT = Path.cwd() if (Path.cwd() / "f1_population.py").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import torch

from f1_node_data import ABLATION_VARIANTS, ablation_schema, build_kg_data, load_population, race_entry_labels, stratified_split
from f1_population import SCHEMA_FILES, read_schema
from node_prediction.gnn_tools import cleanup, limit_cuda_memory_to_free_vram, run_ablation_variant, summarize_ablation
from node_prediction.models import RGCNModel

TUNING_CSV = ROOT / "outputs" / "f1" / "node_prediction" / "rgcn" / "hybrid_sulo_sequential_tuning_results.csv"
ABLATION_CSV = ROOT / "outputs" / "f1" / "node_prediction" / "ablation" / "rgcn" / "ablation_results.csv"
OUT_DIR = ROOT / "outputs" / "f1" / "node_prediction" / "ablation_k2" / "rgcn"
OUT_DIR.mkdir(parents=True, exist_ok=True)
VARIANTS = ["hybrid_sulo", "hybrid_minus_rank_reification", "adhoc", "adhoc_plus_rank_reification"]
print(torch.__version__, "| CUDA:", torch.cuda.is_available())
# WSL2 would otherwise overcommit VRAM into RAM and fail with a driver error instead of an OOM,
# which the run could not skip.
if torch.cuda.is_available():
    print(f"CUDA memory cap: {limit_cuda_memory_to_free_vram() / 2**30:.2f} GiB")

/home/domin/miniconda3/envs/sulo/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


2.7.0+cu126 | CUDA: True
CUDA memory cap: 10.48 GiB


## Data

The four graphs exactly as in the enrichment ablation: rendered without the label and the race outcomes
(`build_kg_data`), with the shared stratified split.

In [2]:
entities, assertions = load_population(ROOT / "data" / "f1")
labels = race_entry_labels(assertions)
split = stratified_split(labels, seed=42)
schemas = {kg: read_schema(str(ROOT / path)) for kg, path in SCHEMA_FILES.items()}
kg_data = {v: build_kg_data(ABLATION_VARIANTS[v][0], entities, assertions, split, ablation_schema(v, schemas))
           for v in VARIANTS}
controls = {v: ABLATION_VARIANTS[v][0] for v in VARIANTS}
pd.DataFrame({v: {"control": controls[v], "nodes": d.data.num_nodes, "edges (fwd + inv)": d.data.edge_index.size(1)}
              for v, d in kg_data.items()}).T

,control,nodes,edges (fwd + inv)
hybrid_sulo,hybrid_sulo,57985,525162
hybrid_minus_rank_reification,hybrid_sulo,57951,525026
adhoc,adhoc,57946,432642
adhoc_plus_rank_reification,adhoc,57981,432778


## Constants

The ablation's constants, with the same sources; the configuration is the Hybrid winner of `RGCN.ipynb`'s
tuning run with one layer fewer.

In [3]:
# Full-batch, so one epoch = one gradient step. Max epochs 400 = the top of GraphGym's design space
# {100, 200, 400} (You et al. 2020, Section 4); Kipf & Welling 2017 cap at 200. Patience 50: no
# literature value (user choice, 2026-10-21). Same values as the main notebooks.
#   https://arxiv.org/abs/2011.08843
#   https://arxiv.org/abs/1609.02907
EPOCHS = 400
PATIENCE = 50

# Early-stopping warm-up (user, 2026-10-29), as in the ablation: epochs 0-49 are trained but neither
# checkpointed nor counted towards patience, as Keras' EarlyStopping(start_from_epoch). No literature value.
#   https://keras.io/api/callbacks/early_stopping/
START_FROM_EPOCH = 50

# Node input embedding size: the GraphSAGE reference implementation recommends learned identity
# features of 64-256 dims (README, --identity_dim).
#   https://github.com/williamleif/GraphSAGE#readme
FEATURE_DIM = 256

# Dropout: PyG's R-GCN example uses p=0.2 between the RGCNConv layers; the original
# R-GCN paper uses no dropout for entity classification.
#   https://github.com/pyg-team/pytorch_geometric/blob/master/examples/rgcn_link_pred.py
DROPOUT = 0.2

# The main notebooks' final-evaluation seeds (user, 2026-10-20); results are mean ± std over them.
FINAL_SEEDS = [0, 1, 2, 3, 4]

# Fixed configuration: the Hybrid winner at K = 2 (its dim, bases, lr and wd; one layer fewer).
winners = pd.read_csv(TUNING_CSV).query("phase == '4_Final_Test_Evaluation'")
config_columns = ["hidden_dim", "layers", "num_bases", "lr", "weight_decay"]
assert len(winners.drop_duplicates(config_columns)) == 1, "phase-4 rows disagree on the configuration"
winner = winners.iloc[0]
assert int(winner["layers"]) == 3, "K = 2 is one layer fewer than a 3-layer winner"
DEPTH = 2
HIDDEN = [int(winner["hidden_dim"])] * (DEPTH - 1)
BASES = None if pd.isna(winner["num_bases"]) else int(winner["num_bases"])
assert BASES is None or all(BASES <= d.num_relations for d in kg_data.values()), "bases > R in a variant"
LR, WD = float(winner["lr"]), float(winner["weight_decay"])
print(f"Hybrid winner at K = 2: hidden {HIDDEN}, bases {BASES}, lr {LR}, wd {WD}")

Hybrid winner at K = 2: hidden [64], bases 10, lr 0.01, wd 0.0001


In [4]:
def model_factory(d):
    """(hidden_list, bases) -> fresh model for one variant; everything variant-specific is bound here."""
    def build(hidden_list, bases):
        return RGCNModel(
            num_keys=len(d.vocabulary), feature_dim=FEATURE_DIM, hidden_channels_list=hidden_list,
            num_relations=d.num_relations, num_bases=bases, dropout=DROPOUT,
        )
    return build

## Runs

Rows are appended to `K2.csv` as each seed finishes: after a crash, finished (variant, seed) pairs are skipped
and failed ones retried. Constants are not checked, so move the CSV away after changing one.

In [5]:
csv_path = OUT_DIR / "K2.csv"
for variant, d in kg_data.items():
    run_ablation_variant(
        model_factory(d), d, variant, HIDDEN, BASES, LR, WD,
        epochs=EPOCHS, patience=PATIENCE, start_from_epoch=START_FROM_EPOCH, seeds=FINAL_SEEDS,
        save_dir=f"{OUT_DIR / 'checkpoints'}/", csv_path=csv_path,
    )
    cleanup()

results = pd.read_csv(csv_path)
failed = results[results["status"] != "ok"]
assert failed.empty, f"Runs that did not finish:\n{failed[['variant', 'seed', 'status']]}"
assert len(results) == len(kg_data) * len(FINAL_SEEDS)
early = results[results["best_epoch"] < START_FROM_EPOCH]
assert early.empty, f"Rows selected inside the warm-up:\n{early[['variant', 'seed', 'best_epoch']]}"


=== Ablation hybrid_sulo | seed 0 ===
Epoch 000 | Train Loss: 1.6747 | Val Loss: 38.5026 | Val Acc: 0.3130 | Val F1: 0.4768 | Val ROC-AUC: 0.5000 | Grad Norm: 12.2748
Epoch 001 | Train Loss: 38.1831 | Val Loss: 14.7650 | Val Acc: 0.3130 | Val F1: 0.4768 | Val ROC-AUC: 0.5382 | Grad Norm: 49.5567
Epoch 002 | Train Loss: 15.0890 | Val Loss: 2.8302 | Val Acc: 0.6870 | Val F1: 0.0000 | Val ROC-AUC: 0.5433 | Grad Norm: 29.2578
Epoch 003 | Train Loss: 2.8282 | Val Loss: 4.0008 | Val Acc: 0.6870 | Val F1: 0.0000 | Val ROC-AUC: 0.4650 | Grad Norm: 10.6147
Epoch 004 | Train Loss: 3.9105 | Val Loss: 3.2877 | Val Acc: 0.6870 | Val F1: 0.0000 | Val ROC-AUC: 0.5427 | Grad Norm: 11.5456
Epoch 005 | Train Loss: 3.2424 | Val Loss: 1.9886 | Val Acc: 0.6870 | Val F1: 0.0000 | Val ROC-AUC: 0.5781 | Grad Norm: 11.0455
Epoch 006 | Train Loss: 2.0256 | Val Loss: 1.0117 | Val Acc: 0.6171 | Val F1: 0.4722 | Val ROC-AUC: 0.5782 | Grad Norm: 9.1742
Epoch 007 | Train Loss: 1.2290 | Val Loss: 1.4560 | Val Acc: 0

## Results

Mean ± std over `FINAL_SEEDS` and the difference from the control (`delta_*`), at K = 2 and, for the same four
graphs, at K = 3 from the ablation (same configuration and protocol, one more layer). Differences are reported as
mean ± std, with no significance test (as in the main notebooks).

In [6]:
summary = summarize_ablation(results, controls)
summary.round(4).to_csv(OUT_DIR / "K2_summary.csv")

k3 = pd.read_csv(ABLATION_CSV).query("variant in @VARIANTS and seed in @FINAL_SEEDS")
assert (k3["layers"] == 3).all() and (k3["hidden_dim"] == HIDDEN[0]).all(), "the ablation ran another configuration"
summary_k3 = summarize_ablation(k3, controls)
pd.concat({"K = 2": summary[["test_roc_auc_mean", "test_roc_auc_std", "delta_test_roc_auc"]],
           "K = 3 (ablation)": summary_k3[["test_roc_auc_mean", "test_roc_auc_std", "delta_test_roc_auc"]]},
          axis=1).round(4)

K = 2                   \
                              test_roc_auc_mean test_roc_auc_std   
variant                                                            
hybrid_sulo                              0.8364           0.0034   
hybrid_minus_rank_reification            0.8675           0.0020   
adhoc                                    0.8684           0.0025   
adhoc_plus_rank_reification              0.8368           0.0012   

                                                  K = 3 (ablation)  \
                              delta_test_roc_auc test_roc_auc_mean   
variant                                                              
hybrid_sulo                               0.0000            0.8690   
hybrid_minus_rank_reification             0.0311            0.8687   
adhoc                                     0.0000            0.8692   
adhoc_plus_rank_reification              -0.0316            0.8686   

                                                                   
                              test_roc_auc_std delta_test_roc_auc  
variant                                                            
hybrid_sulo                             0.0022             0.0000  
hybrid_minus_rank_reification           0.0019            -0.0003  
adhoc                                   0.0017             0.0000  
adhoc_plus_rank_reification             0.0016            -0.0007

## Limitation: runs ended by the epoch cap

A seed whose best epoch is ≥ `EPOCHS − PATIENCE` was cut off by the `EPOCHS` cap while possibly still
improving, so its score may be understated (the same limitation as in the main notebooks).

In [7]:
results[results["best_epoch"] >= EPOCHS - PATIENCE][["variant", "seed", "best_epoch", "val_roc_auc", "test_roc_auc"]]

,variant,seed,best_epoch,val_roc_auc,test_roc_auc
8,hybrid_minus_rank_reification,3,354,0.838768,0.866282
